# 🌡️ Colab 4 — Modelos y Temperature
### Materia: Procesamiento de Lenguaje Natural — Nivel Terciario

---

## La pregunta central de este colab

> Mismo retrieval, mismo prompt. ¿Importa qué modelo usamos y con qué temperature?

En los colabs anteriores exploramos chunking, embeddings y prompts. Ahora llegamos al último componente del pipeline: **el modelo generador**.

Las decisiones acá son:
- ¿Qué modelo uso? (calidad vs costo vs velocidad)
- ¿Con qué temperature? (determinismo vs creatividad)

Y algo que raramente se enseña pero es crítico en producción:
- ¿Cuántos tokens consume cada opción?
- ¿Cuánto cuesta cada llamada?
- ¿Cuánto tarda en responder?

## Las variables que vamos a explorar

| Variable | Opciones |
|---|---|
| **Temperature** | 0.0 / 0.3 / 0.7 / 1.2 |
| **Modelos OpenAI** | gpt-4o-mini / gpt-4o |
| **Tokens y costo** | Medición real de cada llamada |
| **Latencia** | Tiempo de respuesta por modelo |

> 🎯 **Objetivo:** entender que elegir el modelo y la temperature es una decisión de negocio tanto como técnica.

In [4]:
!pip install chromadb openai --quiet
print('✅ Dependencias instaladas')

✅ Dependencias instaladas


In [5]:
import os
import time
from openai import OpenAI

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    api_key = os.getenv('OPENAI_API_KEY', 'TU_API_KEY_ACA')

client = OpenAI(api_key=api_key)
print('✅ Cliente OpenAI inicializado')

✅ Cliente OpenAI inicializado


---
## 📄 Setup: corpus, retrieval y prompt fijos

En este colab **nada cambia excepto el modelo y la temperature**.
Así podemos aislar el efecto de cada variable.

In [6]:
import chromadb
from chromadb.utils import embedding_functions

CHUNKS = [
    "Los clientes tienen derecho a solicitar la devolución de productos dentro de los 30 días corridos desde la fecha de entrega. Para productos con defectos de fábrica, el plazo se extiende a 90 días. Las devoluciones fuera de estos plazos no serán aceptadas salvo orden judicial.",
    "El producto debe presentarse en su embalaje original, sin señales de uso, con todos los accesorios incluidos y con el ticket o factura de compra. Los productos de higiene personal, auriculares in-ear y colchones no admiten devolución por razones sanitarias.",
    "El cliente debe iniciar el proceso de devolución a través del portal web en la sección Mi Cuenta. Una vez aprobada la solicitud recibirá un código RMA. Las devoluciones sin RMA serán rechazadas. El reembolso se acredita dentro de los 10 días hábiles.",
    "Todos los productos tienen garantía legal mínima de 6 meses por defectos de fabricación, conforme a la Ley 24.240 de Defensa del Consumidor.",
    "Los televisores de 40 pulgadas o más tienen garantía de 12 meses. Las notebooks, tablets y computadoras tienen garantía de 12 meses. Los smartphones de gama alta tienen garantía de 12 meses. Los accesorios y cables tienen garantía de 3 meses.",
    "La garantía no cubre daños por mal uso, golpes, líquidos, modificaciones no autorizadas o uso fuera de las condiciones del fabricante.",
    "Para hacer efectiva la garantía el cliente debe presentar el producto en cualquier sucursal. El tiempo de reparación es de 15 días hábiles. Si no puede completarse, TechStore ofrecerá reemplazo o reembolso.",
    "El envío es gratuito para compras iguales o superiores a $50.000. Para compras menores el costo se calcula según peso y distancia.",
    "Para CABA y GBA el plazo es de 24 a 48 horas hábiles. Para el interior es de 3 a 7 días hábiles. Patagonia y NOA tienen un plazo de 7 a 12 días hábiles.",
    "Se aceptan tarjetas Visa, Mastercard, American Express y Cabal. También Mercado Pago, Modo, Ualá y Naranja X. Las tarjetas prepagas solo se aceptan hasta $30.000.",
    "Con Visa y Mastercard se ofrecen 3, 6 y 12 cuotas sin interés en productos seleccionados. Los productos en liquidación no participan. Las cuotas no son acumulables con otros descuentos.",
    "Los pagos por transferencia bancaria reciben un descuento del 10%. La transferencia debe acreditarse en 48 horas o la orden se cancela.",
    "El centro de atención opera de lunes a viernes de 9 a 18 hs y sábados de 9 a 13 hs. Canales: WhatsApp +54 11 4000-0000, email soporte@techstore.com.ar, chat en vivo y atención presencial.",
    "Los reclamos formales se resuelven en máximo 72 horas hábiles. En casos complejos el plazo puede extenderse hasta 10 días hábiles notificando al cliente.",
    "Si el cliente no está satisfecho puede escalar al área de Supervisión o presentar el reclamo ante la Dirección de Defensa del Consumidor de su provincia.",
]

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=api_key,
    model_name="text-embedding-3-small"
)
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("colab4")
except:
    pass
coleccion = chroma_client.create_collection("colab4", embedding_function=openai_ef)
coleccion.add(documents=CHUNKS, ids=[f"c{i}" for i in range(len(CHUNKS))])

def recuperar_contexto(pregunta, top_k=3):
    res = coleccion.query(query_texts=[pregunta], n_results=top_k)
    return "\n\n".join(res['documents'][0])

# System prompt fijo — el mismo de producción del Colab 3
SYSTEM_PROMPT = """Sos Tomi, el asistente virtual de TechStore Argentina.
Respondé ÚNICAMENTE con información del contexto provisto.
Si la información no está en el contexto, decí: "Esa consulta requiere atención personalizada. Contactanos por WhatsApp al +54 11 4000-0000."
Usá español argentino. Sé conciso y amable. No menciones que tenés un contexto."""

print(f"✅ Setup completo")
print(f"   Chunks indexados: {coleccion.count()}")
print(f"   System prompt: fijo")
print(f"   Retrieval: fijo (top-3)")
print(f"   Variable 1: temperature")
print(f"   Variable 2: modelo")

✅ Setup completo
   Chunks indexados: 15
   System prompt: fijo
   Retrieval: fijo (top-3)
   Variable 1: temperature
   Variable 2: modelo


---
## 💰 Precios de referencia (junio 2025)

Antes de experimentar, conviene tener presentes los costos aproximados:

| Modelo | Input (por 1M tokens) | Output (por 1M tokens) | Perfil |
|---|---|---|---|
| **gpt-4o-mini** | $0.15 | $0.60 | Rápido, económico, muy bueno para tareas estructuradas |
| **gpt-4o** | $2.50 | $10.00 | Más potente, mejor razonamiento complejo, ~16x más caro |

> 💡 Para un sistema de atención al cliente con 10.000 consultas diarias de ~500 tokens cada una, la diferencia entre gpt-4o-mini y gpt-4o puede ser **miles de dólares por mes**.

In [7]:
# Función de llamada con métricas completas
def llamar_con_metricas(pregunta, modelo, temperature, verbose=True):
    contexto  = recuperar_contexto(pregunta)
    user_prompt = f"CONTEXTO:\n{contexto}\n\nCONSULTA: {pregunta}"

    # Precios por millón de tokens (junio 2025)
    precios = {
        "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
        "gpt-4o":      {"input": 2.50,  "output": 10.00},
    }

    inicio = time.time()
    response = client.chat.completions.create(
        model=modelo,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt}
        ],
        temperature=temperature,
        max_tokens=300
    )
    latencia = time.time() - inicio

    tok_in  = response.usage.prompt_tokens
    tok_out = response.usage.completion_tokens
    precio  = precios.get(modelo, {"input": 0, "output": 0})
    costo   = (tok_in * precio["input"] + tok_out * precio["output"]) / 1_000_000
    texto   = response.choices[0].message.content

    if verbose:
        print(f"  🤖 Modelo:      {modelo}")
        print(f"  🌡️  Temperature: {temperature}")
        print(f"  📥 Tokens in:   {tok_in}")
        print(f"  📤 Tokens out:  {tok_out}")
        print(f"  💰 Costo:       ${costo:.6f} USD")
        print(f"  ⏱️  Latencia:    {latencia:.2f}s")
        print(f"  💬 Respuesta:   {texto}")
        print()

    return {
        "modelo": modelo, "temperature": temperature,
        "tok_in": tok_in, "tok_out": tok_out,
        "costo": costo, "latencia": latencia, "respuesta": texto
    }

print("✅ Función lista. Cada llamada mide tokens, costo y latencia.")

✅ Función lista. Cada llamada mide tokens, costo y latencia.


---
# 🔬 Experimento 1 — El efecto de la temperature

Recordemos del Bloque 1: la temperature escala los logits antes del Softmax.
- **Temperature 0:** siempre el token más probable → respuestas deterministas
- **Temperature alta:** distribución más plana → más variedad, más creatividad

Para un chatbot de atención al cliente, ¿qué temperature tiene sentido?

In [8]:
PREGUNTA = "Compré una notebook hace 8 meses y se rompió. ¿Tengo garantía?"

temperatures = [0.0, 0.3, 0.7, 1.2]

print(f"❓ Pregunta: {PREGUNTA}")
print(f"   Modelo fijo: gpt-4o-mini")
print("=" * 70)

resultados_temp = []
for temp in temperatures:
    print(f"\n🌡️  Temperature = {temp}")
    print("-" * 40)
    r = llamar_con_metricas(PREGUNTA, "gpt-4o-mini", temp)
    resultados_temp.append(r)

❓ Pregunta: Compré una notebook hace 8 meses y se rompió. ¿Tengo garantía?
   Modelo fijo: gpt-4o-mini

🌡️  Temperature = 0.0
----------------------------------------
  🤖 Modelo:      gpt-4o-mini
  🌡️  Temperature: 0.0
  📥 Tokens in:   248
  📤 Tokens out:  33
  💰 Costo:       $0.000057 USD
  ⏱️  Latencia:    5.19s
  💬 Respuesta:   Sí, tenés garantía. Las notebooks tienen una garantía de 12 meses, así que podés hacer efectiva la garantía presentando el producto en cualquier sucursal.


🌡️  Temperature = 0.3
----------------------------------------
  🤖 Modelo:      gpt-4o-mini
  🌡️  Temperature: 0.3
  📥 Tokens in:   248
  📤 Tokens out:  32
  💰 Costo:       $0.000056 USD
  ⏱️  Latencia:    1.48s
  💬 Respuesta:   Sí, tenés garantía. Las notebooks tienen garantía de 12 meses, así que podés presentar el producto en cualquier sucursal para hacer efectiva la garantía.


🌡️  Temperature = 0.7
----------------------------------------
  🤖 Modelo:      gpt-4o-mini
  🌡️  Temperature: 0.7
  📥 Tokens

In [9]:
# Mostramos variabilidad: misma temperature, misma pregunta, tres veces
# Con temperature 0 debería ser idéntico; con 0.7 debería variar

print("🔄 Variabilidad: 3 llamadas con la misma configuración")
print("=" * 70)

for temp in [0.0, 0.7]:
    print(f"\n🌡️  Temperature = {temp} (3 repeticiones):")
    respuestas = []
    for i in range(3):
        r = llamar_con_metricas(PREGUNTA, "gpt-4o-mini", temp, verbose=False)
        respuestas.append(r["respuesta"])
        print(f"  [{i+1}] {r['respuesta'][:120]}...")

    # ¿Son todas iguales?
    todas_iguales = all(r == respuestas[0] for r in respuestas)
    print(f"  → ¿Respuestas idénticas? {'✅ Sí' if todas_iguales else '❌ No, varían'}")

🔄 Variabilidad: 3 llamadas con la misma configuración

🌡️  Temperature = 0.0 (3 repeticiones):
  [1] Sí, tenés garantía. Las notebooks tienen una garantía de 12 meses, así que podés hacer efectiva la garantía presentando ...
  [2] Sí, tenés garantía. Las notebooks tienen una garantía de 12 meses, así que podés hacer efectiva la garantía presentando ...
  [3] Sí, tenés garantía. Las notebooks tienen una garantía de 12 meses, así que podés hacer efectiva la garantía presentando ...
  → ¿Respuestas idénticas? ✅ Sí

🌡️  Temperature = 0.7 (3 repeticiones):
  [1] Sí, tenés garantía para tu notebook, ya que todos los productos tienen una garantía de 12 meses. Podés presentar el prod...
  [2] Sí, tenés garantía por tu notebook, ya que cuenta con 12 meses de garantía. Podés presentar el producto en cualquier suc...
  [3] Sí, tenés garantía. Las notebooks tienen una garantía de 12 meses, por lo que podés hacer efectiva la garantía presentan...
  → ¿Respuestas idénticas? ❌ No, varían


### 🔍 ¿Qué observás?

Con **temperature 0** las respuestas son deterministas: la misma pregunta siempre da la misma respuesta. Ideal para atención al cliente donde la consistencia es clave.

Con **temperature alta** las respuestas varían. Puede ser útil para generación creativa, pero para un chatbot de soporte genera inconsistencias que confunden al cliente.

> 💡 **Regla práctica:** para RAG de atención al cliente usá temperature entre 0.0 y 0.3. Reservá temperatures altas para generación creativa.

---
# 🔬 Experimento 2 — Comparación de modelos

Mismo retrieval, mismo prompt, misma temperature. Solo cambia el modelo.
¿Justifica la diferencia de calidad el costo adicional?

In [10]:
# Preguntas de distintos niveles de complejidad
preguntas = [
    # Simple: respuesta directa en un chunk
    "¿Cuál es el horario de atención al cliente?",
    # Media: requiere combinar 2 chunks
    "Pagué con transferencia. ¿Cuándo me llega el pedido a Mar del Plata?",
    # Compleja: razonamiento multi-paso
    "Compré un televisor de 55 pulgadas hace 14 meses con tarjeta de crédito. Dejó de funcionar. ¿Tengo algún recurso?",
]

modelos = ["gpt-4o-mini", "gpt-4o"]
TEMPERATURE_FIJA = 0.2

resumen = []

for pregunta in preguntas:
    print(f"\n❓ {pregunta}")
    print("=" * 70)
    for modelo in modelos:
        r = llamar_con_metricas(pregunta, modelo, TEMPERATURE_FIJA)
        resumen.append({**r, "pregunta": pregunta[:50]})


❓ ¿Cuál es el horario de atención al cliente?
  🤖 Modelo:      gpt-4o-mini
  🌡️  Temperature: 0.2
  📥 Tokens in:   254
  📤 Tokens out:  29
  💰 Costo:       $0.000056 USD
  ⏱️  Latencia:    0.98s
  💬 Respuesta:   El horario de atención al cliente es de lunes a viernes de 9 a 18 hs y sábados de 9 a 13 hs.

  🤖 Modelo:      gpt-4o
  🌡️  Temperature: 0.2
  📥 Tokens in:   254
  📤 Tokens out:  29
  💰 Costo:       $0.000925 USD
  ⏱️  Latencia:    0.78s
  💬 Respuesta:   El horario de atención al cliente es de lunes a viernes de 9 a 18 hs y sábados de 9 a 13 hs.


❓ Pagué con transferencia. ¿Cuándo me llega el pedido a Mar del Plata?
  🤖 Modelo:      gpt-4o-mini
  🌡️  Temperature: 0.2
  📥 Tokens in:   240
  📤 Tokens out:  53
  💰 Costo:       $0.000068 USD
  ⏱️  Latencia:    1.11s
  💬 Respuesta:   Para Mar del Plata, el plazo de entrega es de 3 a 7 días hábiles una vez que la transferencia se acredite. Recuerda que la transferencia debe acreditarse en 48 horas, de lo contrario, la orden se canc

In [11]:
# Tabla comparativa de costos y latencia
print("\n📊 TABLA COMPARATIVA")
print("=" * 90)
print(f"{'Pregunta':<45} {'Modelo':<15} {'Tokens':<8} {'Costo USD':<12} {'Latencia'}")
print("-" * 90)

for r in resumen:
    print(f"{r['pregunta']:<45} {r['modelo']:<15} {r['tok_in']+r['tok_out']:<8} ${r['costo']:<11.6f} {r['latencia']:.2f}s")

# Totales por modelo
for modelo in modelos:
    rows = [r for r in resumen if r['modelo'] == modelo]
    costo_total   = sum(r['costo'] for r in rows)
    latencia_prom = sum(r['latencia'] for r in rows) / len(rows)
    print(f"\n  {modelo}: costo total = ${costo_total:.6f} | latencia promedio = {latencia_prom:.2f}s")


📊 TABLA COMPARATIVA
Pregunta                                      Modelo          Tokens   Costo USD    Latencia
------------------------------------------------------------------------------------------
¿Cuál es el horario de atención al cliente?   gpt-4o-mini     283      $0.000056    0.98s
¿Cuál es el horario de atención al cliente?   gpt-4o          283      $0.000925    0.78s
Pagué con transferencia. ¿Cuándo me llega el pedid gpt-4o-mini     293      $0.000068    1.11s
Pagué con transferencia. ¿Cuándo me llega el pedid gpt-4o          267      $0.000870    0.65s
Compré un televisor de 55 pulgadas hace 14 meses c gpt-4o-mini     307      $0.000067    1.63s
Compré un televisor de 55 pulgadas hace 14 meses c gpt-4o          284      $0.000890    0.63s

  gpt-4o-mini: costo total = $0.000190 | latencia promedio = 1.24s

  gpt-4o: costo total = $0.002685 | latencia promedio = 0.69s


### 🔍 ¿Cuándo vale la pena pagar más?

Observá en la pregunta compleja (el televisor fuera de garantía):
- ¿Algún modelo detecta que 14 meses supera la garantía estándar pero considera la garantía extendida?
- ¿Alguno sugiere proactivamente escalar el reclamo?

Para preguntas simples (horarios, envíos) **gpt-4o-mini es más que suficiente**.
Para razonamiento complejo que cruza múltiples condiciones, gpt-4o puede dar respuestas más completas.

> 💡 **Estrategia híbrida:** clasificar la consulta primero (simple/compleja) y rutear a distintos modelos según el caso. Reduce costos sin sacrificar calidad.

---
# 🔬 Experimento 3 — Proyección de costos a escala

Ahora hacemos algo que casi nunca se enseña: proyectar el costo real de operar un sistema RAG en producción.

In [12]:
# Tomamos el promedio de tokens de las llamadas anteriores
import statistics

for modelo in modelos:
    rows = [r for r in resumen if r['modelo'] == modelo]
    tok_promedio = statistics.mean([r['tok_in'] + r['tok_out'] for r in rows])
    costo_promedio = statistics.mean([r['costo'] for r in rows])

    print(f"\n📊 Proyección para: {modelo}")
    print(f"   Tokens promedio por consulta: {tok_promedio:.0f}")
    print(f"   Costo promedio por consulta:  ${costo_promedio:.6f} USD")
    print()

    escenarios = [
        ("Proyecto universitario",     100,    30),
        ("Startup pequeña",          1_000,    30),
        ("Empresa mediana",         10_000,    30),
        ("E-commerce grande",      100_000,    30),
    ]

    print(f"  {'Escenario':<25} {'Consultas/día':<15} {'Costo/mes USD'}")
    print(f"  {'-'*60}")
    for nombre, consultas_dia, dias in escenarios:
        costo_mes = consultas_dia * dias * costo_promedio
        print(f"  {nombre:<25} {consultas_dia:<15,} ${costo_mes:,.2f}")


📊 Proyección para: gpt-4o-mini
   Tokens promedio por consulta: 294
   Costo promedio por consulta:  $0.000063 USD

  Escenario                 Consultas/día   Costo/mes USD
  ------------------------------------------------------------
  Proyecto universitario    100             $0.19
  Startup pequeña           1,000           $1.91
  Empresa mediana           10,000          $19.05
  E-commerce grande         100,000         $190.50

📊 Proyección para: gpt-4o
   Tokens promedio por consulta: 278
   Costo promedio por consulta:  $0.000895 USD

  Escenario                 Consultas/día   Costo/mes USD
  ------------------------------------------------------------
  Proyecto universitario    100             $2.69
  Startup pequeña           1,000           $26.85
  Empresa mediana           10,000          $268.50
  E-commerce grande         100,000         $2,685.00


### 🔍 Lo que muestra esta tabla

Para un proyecto universitario o una startup chica, ambos modelos son accesibles.
Para un e-commerce grande con 100.000 consultas diarias, la diferencia entre modelos puede ser **decenas de miles de dólares por mes**.

Esto explica por qué en producción:
- Se usan modelos pequeños para la mayoría de las consultas
- Se cachean respuestas frecuentes (caching semántico del Bloque 1)
- Se diseña el prompt para minimizar tokens de salida

---
# 🔬 Experimento 4 — El efecto del top-k en tokens y costo

Cuántos chunks recuperamos (top-k) afecta directamente los tokens de entrada y por ende el costo. Veamos el trade-off.

In [13]:
PREGUNTA = "Compré una notebook con tarjeta. ¿Cuántas cuotas puedo hacer y qué garantía tiene?"

print(f"❓ Pregunta: {PREGUNTA}")
print(f"   Modelo fijo: gpt-4o-mini | Temperature: 0.2")
print("=" * 70)

for top_k in [1, 2, 3, 5]:
    contexto    = recuperar_contexto(PREGUNTA, top_k=top_k)
    user_prompt = f"CONTEXTO:\n{contexto}\n\nCONSULTA: {PREGUNTA}"

    inicio = time.time()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt}
        ],
        temperature=0.2,
        max_tokens=300
    )
    latencia = time.time() - inicio

    tok_in   = response.usage.prompt_tokens
    tok_out  = response.usage.completion_tokens
    costo    = (tok_in * 0.15 + tok_out * 0.60) / 1_000_000
    respuesta = response.choices[0].message.content

    print(f"\n📦 top_k = {top_k}")
    print(f"   Chars de contexto: {len(contexto)} | Tokens in: {tok_in} | Costo: ${costo:.6f}")
    print(f"   Respuesta: {respuesta[:200]}...")

❓ Pregunta: Compré una notebook con tarjeta. ¿Cuántas cuotas puedo hacer y qué garantía tiene?
   Modelo fijo: gpt-4o-mini | Temperature: 0.2

📦 top_k = 1
   Chars de contexto: 242 | Tokens in: 174 | Costo: $0.000049
   Respuesta: La notebook tiene garantía de 12 meses. Respecto a las cuotas, esa consulta requiere atención personalizada. Contactanos por WhatsApp al +54 11 4000-0000....

📦 top_k = 2
   Chars de contexto: 429 | Tokens in: 214 | Costo: $0.000051
   Respuesta: Podés hacer 3, 6 o 12 cuotas sin interés, dependiendo de la promoción vigente. La notebook tiene una garantía de 12 meses....

📦 top_k = 3
   Chars de contexto: 571 | Tokens in: 245 | Costo: $0.000054
   Respuesta: Podés hacer 3, 6 o 12 cuotas sin interés en productos seleccionados. La notebook tiene una garantía de 12 meses....

📦 top_k = 5
   Chars de contexto: 915 | Tokens in: 317 | Costo: $0.000064
   Respuesta: Podés hacer 3, 6 o 12 cuotas sin interés en productos seleccionados. La notebook tiene garantía de 12 

### 🔍 El trade-off del top-k

- **top-k bajo (1-2):** menos tokens, más barato, pero puede perderse información relevante
- **top-k alto (4-5):** más contexto disponible, pero más tokens y más "ruido" para el modelo

Para preguntas simples, top-k=2 suele ser suficiente.
Para preguntas que cruzan múltiples temas (cuotas + garantía), top-k=3 o 4 puede ser necesario.

> 💡 En producción avanzada se usa **top-k dinámico**: se clasifica la consulta y se elige el top-k según la complejidad estimada.

---
# 🔬 Experimento 5 — Transparencia total: el prompt completo

Acá mostramos exactamente qué le llega al modelo. Esto es lo mismo que podés ver en la consola de OpenAI Platform.

> 💡 En la consola de platform.openai.com podés ver el prompt completo, los tokens y el costo de cada llamada en tiempo real. Es muy útil para debuggear un sistema RAG.

In [14]:
PREGUNTA = "¿Puedo devolver unos auriculares que usé una vez?"

contexto    = recuperar_contexto(PREGUNTA, top_k=3)
user_prompt = f"CONTEXTO:\n{contexto}\n\nCONSULTA: {PREGUNTA}"

print("=" * 70)
print("📋 PAYLOAD COMPLETO QUE RECIBE EL MODELO")
print("   (exactamente lo que verías en OpenAI Platform)")
print("=" * 70)

print("\n[SYSTEM MESSAGE]")
print("-" * 40)
print(SYSTEM_PROMPT)

print("\n[USER MESSAGE]")
print("-" * 40)
print(user_prompt)

print("\n" + "=" * 70)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt}
    ],
    temperature=0.2,
    max_tokens=300
)

tok_in  = response.usage.prompt_tokens
tok_out = response.usage.completion_tokens
costo   = (tok_in * 0.15 + tok_out * 0.60) / 1_000_000

print("\n[ASSISTANT MESSAGE]")
print("-" * 40)
print(response.choices[0].message.content)

print("\n" + "=" * 70)
print(f"📊 MÉTRICAS")
print(f"   Tokens de entrada (system + contexto + pregunta): {tok_in}")
print(f"   Tokens de salida (respuesta):                     {tok_out}")
print(f"   Total tokens:                                     {tok_in + tok_out}")
print(f"   Costo de esta llamada:                            ${costo:.6f} USD")
print(f"   Costo si se repite 10.000 veces/día:              ${costo * 10000 * 30:.2f} USD/mes")

📋 PAYLOAD COMPLETO QUE RECIBE EL MODELO
   (exactamente lo que verías en OpenAI Platform)

[SYSTEM MESSAGE]
----------------------------------------
Sos Tomi, el asistente virtual de TechStore Argentina.
Respondé ÚNICAMENTE con información del contexto provisto.
Si la información no está en el contexto, decí: "Esa consulta requiere atención personalizada. Contactanos por WhatsApp al +54 11 4000-0000."
Usá español argentino. Sé conciso y amable. No menciones que tenés un contexto.

[USER MESSAGE]
----------------------------------------
CONTEXTO:
El producto debe presentarse en su embalaje original, sin señales de uso, con todos los accesorios incluidos y con el ticket o factura de compra. Los productos de higiene personal, auriculares in-ear y colchones no admiten devolución por razones sanitarias.

Los clientes tienen derecho a solicitar la devolución de productos dentro de los 30 días corridos desde la fecha de entrega. Para productos con defectos de fábrica, el plazo se extiende a 9

---
# 🔬 Experimento 6 — Mapa de decisión: ¿qué modelo y temperature elegir?

Cerramos con un experimento integrador que ayuda a tomar la decisión.

In [15]:
# Matriz de decisión: tipo de consulta x modelo
casos = [
    {
        "tipo": "FAQ simple",
        "pregunta": "¿Cuál es el horario de atención?",
        "temperatura_recomendada": 0.0,
        "modelo_recomendado": "gpt-4o-mini",
        "razon": "Respuesta única y determinista"
    },
    {
        "tipo": "Consulta de políticas",
        "pregunta": "¿Qué condiciones necesito para devolver un producto?",
        "temperatura_recomendada": 0.2,
        "modelo_recomendado": "gpt-4o-mini",
        "razon": "Respuesta estructurada, bajo costo"
    },
    {
        "tipo": "Caso complejo multi-condición",
        "pregunta": "Compré hace 13 meses una notebook de $350.000. Se rompió la pantalla por un golpe. ¿Tengo garantía? ¿Puedo reclamar algo?",
        "temperatura_recomendada": 0.2,
        "modelo_recomendado": "gpt-4o",
        "razon": "Requiere razonamiento sobre múltiples condiciones"
    },
]

for caso in casos:
    print(f"\n📋 Tipo: {caso['tipo']}")
    print(f"   Recomendación: {caso['modelo_recomendado']} | temp={caso['temperatura_recomendada']}")
    print(f"   Razón: {caso['razon']}")
    print(f"   ❓ {caso['pregunta']}")
    print("-" * 60)

    r = llamar_con_metricas(
        caso["pregunta"],
        caso["modelo_recomendado"],
        caso["temperatura_recomendada"]
    )


📋 Tipo: FAQ simple
   Recomendación: gpt-4o-mini | temp=0.0
   Razón: Respuesta única y determinista
   ❓ ¿Cuál es el horario de atención?
------------------------------------------------------------
  🤖 Modelo:      gpt-4o-mini
  🌡️  Temperature: 0.0
  📥 Tokens in:   252
  📤 Tokens out:  27
  💰 Costo:       $0.000054 USD
  ⏱️  Latencia:    0.84s
  💬 Respuesta:   El horario de atención es de lunes a viernes de 9 a 18 hs y sábados de 9 a 13 hs.


📋 Tipo: Consulta de políticas
   Recomendación: gpt-4o-mini | temp=0.2
   Razón: Respuesta estructurada, bajo costo
   ❓ ¿Qué condiciones necesito para devolver un producto?
------------------------------------------------------------
  🤖 Modelo:      gpt-4o-mini
  🌡️  Temperature: 0.2
  📥 Tokens in:   278
  📤 Tokens out:  110
  💰 Costo:       $0.000108 USD
  ⏱️  Latencia:    2.04s
  💬 Respuesta:   Para devolver un producto, debes cumplir con las siguientes condiciones:

1. El producto debe estar en su embalaje original.
2. No debe mostrar señ

---
# 🧪 Zona de experimentación libre

In [16]:
# ✏️ Probá tu propia combinación de modelo y temperature

MI_PREGUNTA    = "¿Qué pasa si me llega un producto dañado?"
MI_MODELO      = "gpt-4o-mini"   # o "gpt-4o"
MI_TEMPERATURE = 0.5             # entre 0.0 y 2.0

print(f"❓ Pregunta:    {MI_PREGUNTA}")
print(f"🤖 Modelo:      {MI_MODELO}")
print(f"🌡️  Temperature: {MI_TEMPERATURE}")
print("=" * 60)

r = llamar_con_metricas(MI_PREGUNTA, MI_MODELO, MI_TEMPERATURE)

❓ Pregunta:    ¿Qué pasa si me llega un producto dañado?
🤖 Modelo:      gpt-4o-mini
🌡️  Temperature: 0.5
  🤖 Modelo:      gpt-4o-mini
  🌡️  Temperature: 0.5
  📥 Tokens in:   268
  📤 Tokens out:  24
  💰 Costo:       $0.000055 USD
  ⏱️  Latencia:    0.81s
  💬 Respuesta:   Esa consulta requiere atención personalizada. Contactanos por WhatsApp al +54 11 4000-0000.



In [17]:
# 🔄 Comparador rápido: misma pregunta, todos los modelos disponibles

MI_PREGUNTA = "Pagué con transferencia hace 2 días y no me confirmaron el pedido."

print(f"❓ {MI_PREGUNTA}\n")
print(f"{'Modelo':<15} {'Temp':<6} {'Tokens':<8} {'Costo USD':<12} {'Respuesta (primeros 100 chars)'}")
print("=" * 90)

for modelo in ["gpt-4o-mini", "gpt-4o"]:
    for temp in [0.0, 0.7]:
        r = llamar_con_metricas(MI_PREGUNTA, modelo, temp, verbose=False)
        print(f"{r['modelo']:<15} {temp:<6} {r['tok_in']+r['tok_out']:<8} ${r['costo']:<11.6f} {r['respuesta'][:100]}...")

❓ Pagué con transferencia hace 2 días y no me confirmaron el pedido.

Modelo          Temp   Tokens   Costo USD    Respuesta (primeros 100 chars)
gpt-4o-mini     0.0    286      $0.000068    La transferencia debe acreditarse en 48 horas. Si ya pasaron 2 días y no recibiste la confirmación, ...
gpt-4o-mini     0.7    317      $0.000087    Si pagaste con transferencia hace 2 días y no recibiste la confirmación, es posible que la acreditac...
gpt-4o          0.0    254      $0.000815    Esa consulta requiere atención personalizada. Contactanos por WhatsApp al +54 11 4000-0000....
gpt-4o          0.7    287      $0.001145    Verificá si la transferencia se acreditó correctamente. Recordá que debe acreditarse en 48 horas o l...


---
# ✅ Resumen del Colab 4

## Temperature

| Rango | Uso recomendado |
|---|---|
| **0.0** | FAQ, respuestas únicas, máxima consistencia |
| **0.1 - 0.3** | Atención al cliente, QA sobre documentos |
| **0.5 - 0.7** | Respuestas con algo de variedad, redacción |
| **1.0+** | Creatividad, brainstorming, generación libre |

## Modelo

| Modelo | Cuándo usarlo |
|---|---|
| **gpt-4o-mini** | 90% de los casos: FAQs, políticas, consultas estructuradas |
| **gpt-4o** | Razonamiento complejo, múltiples condiciones, casos edge |

## Las tres variables que siempre hay que medir

1. **Calidad de la respuesta** — ¿responde lo que se preguntó?
2. **Costo por consulta** — ¿es sostenible a la escala proyectada?
3. **Latencia** — ¿el tiempo de respuesta es aceptable para el usuario?

---

# 🎉 Completaste los 4 Colabs

| Colab | Variable estudiada | Conclusión clave |
|---|---|---|
| **1 — Chunking** | Tamaño, overlap, estrategia | No hay tamaño universal; depende del documento |
| **2 — Embeddings** | TF-IDF vs semántico | Los embeddings entienden sinónimos; TF-IDF es preciso en términos exactos |
| **3 — Prompt** | Instrucciones, formato, CoT | El prompt es una decisión de diseño tan importante como el modelo |
| **4 — Modelos** | Temperature, modelo, costo | El modelo y la temperature tienen impacto directo en costo y consistencia |

## 🚀 Siguiente paso: el Proyecto Integrador

Tres RAGs en progresión, interfaz Gradio, deployable en HuggingFace Spaces:
- **RAG 1:** Chatbot de políticas de empresa
- **RAG 2:** Control de stock en tiempo real
- **RAG 3:** Orquestador con alertas automáticas